# Predicción de Fallos y Análisis de Data Drift

## 1. Carga de Datos y Preprocesamiento

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Cargar datos
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Preprocesamiento básico
def preprocesar(df_tr, df_ts):
    # Unir para codificar categóricas consistente
    df = pd.concat([df_tr, df_ts], axis=0, ignore_index=True)
    
    # Codificar categóricas
    cols_cat = ['product_code', 'attribute_0', 'attribute_1']
    le = LabelEncoder()
    for col in cols_cat:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))
            
    # Imputar nulos (mediana)
    df = df.fillna(df.median())
    
    # Separar de nuevo
    n_train = len(df_tr)
    return df.iloc[:n_train].copy(), df.iloc[n_train:].copy()

# Preparamos los datos para la Tarea 1
X = df_train.drop(['id', 'failure'], axis=1)
y = df_train['failure']
X_test = df_test.drop(['id'], axis=1)

X, X_test = preprocesar(X, X_test)

## TASK 1: Predict Product Failures
Entrenamos un modelo para predecir si un producto fallará.

In [19]:
# Dividimos train para validar y sacar métricas
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entrenar modelo
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train_split, y_train_split)

# Predicciones
preds_val = model.predict(X_val_split)
preds_proba_val = model.predict_proba(X_val_split)[:, 1]

# --- MÉTRICAS REQUERIDAS ---
print("Accuracy:", accuracy_score(y_val_split, preds_val))
print("ROC-AUC:", roc_auc_score(y_val_split, preds_proba_val))
print("\nReporte de Clasificación:\n", classification_report(y_val_split, preds_val))

# Predicción final para el fichero test.csv (opcional, pero parte de la tarea)
model.fit(X, y)
final_preds = model.predict_proba(X_test)[:, 1]

Accuracy: 0.7719232216785848
ROC-AUC: 0.5252914600923873

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.79      0.97      0.87      4184
           1       0.26      0.04      0.07      1130

    accuracy                           0.77      5314
   macro avg       0.52      0.50      0.47      5314
weighted avg       0.68      0.77      0.70      5314



## TASK 2: Evaluate the data drift to confirm if your model is up-to-date
Pasos indicados:
1. Combinar train y test.
2. Eliminar `failure` de train.
3. Eliminar `id` de ambos.
4. Añadir columna target (0 para train, 1 para test).
5. Entrenar modelo para clasificar si es train o test.

In [20]:
# 1, 2 y 3. Preparar datasets borrando columnas
train_drift = df_train.drop(['id', 'failure'], axis=1).copy()
test_drift = df_test.drop(['id'], axis=1).copy()

# 4. Añadir target
train_drift['target'] = 0
test_drift['target'] = 1

# Combinar
df_drift = pd.concat([train_drift, test_drift], ignore_index=True)

# Preprocesar categóricas para que el modelo funcione
for col in ['product_code', 'attribute_0', 'attribute_1']:
    if col in df_drift.columns:
        df_drift[col] = LabelEncoder().fit_transform(df_drift[col].astype(str))
df_drift = df_drift.fillna(df_drift.median())

# Separar X e y para Data Drift
X_drift = df_drift.drop('target', axis=1)
y_drift = df_drift['target']

# 5. Entrenar modelo para predecir si es train (0) o test (1)
model_drift = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Evaluamos con Cross-Validation usando ROC-AUC
cv_scores = cross_val_score(model_drift, X_drift, y_drift, cv=5, scoring='roc_auc')

print(f"ROC-AUC Medio (Data Drift): {cv_scores.mean():.4f}")

ROC-AUC Medio (Data Drift): 1.0000


### How would this model behave if there is data drift? And in the opposite case?

*   **Si hay Data Drift (Caso actual, AUC $\approx$ 1.0):**
    El modelo es capaz de distinguir perfectamente entre los datos de entrenamiento y los de test. Esto significa que las distribuciones son muy diferentes. En este caso concreto, los `product_code` son distintos entre train y test, lo que hace que sean fácilmente distinguibles.

*   **Si NO hay Data Drift (Caso ideal, AUC $\approx$ 0.5):**
    El modelo se comportaría como un clasificador aleatorio. No sería capaz de distinguir si un registro viene de train o de test, lo cual es bueno porque indicaría que ambos conjuntos de datos son homogéneos y siguen la misma distribución.